# Neuro Move — EEG Image Preprocessing

This notebook accepts an **EEG waveform image as manual input** and produces:

1. Original EEG image
2. Mu-filtered image (8–13 Hz)
3. Beta-filtered image (13–30 Hz)
4. Final preprocessed image (8–30 Hz)

**Important:** An EEG image does not contain its original sampling frequency. Enter the known sampling frequency when prompted. For PhysioNet EEGMMIDB-derived images, use **160 Hz**.


In [ ]:
!pip -q install numpy scipy matplotlib pillow


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from scipy.signal import butter, filtfilt

from google.colab import files

print("Libraries loaded successfully.")


## 1. Upload the EEG image

Run the next cell and select your EEG waveform image from your device.


In [ ]:
print("Please upload your EEG waveform image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)


In [ ]:
img = Image.open(filename).convert("RGB")

plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.title("INPUT: Original EEG Image", fontsize=16)
plt.axis("off")
plt.show()


## 2. Enter the sampling frequency

For a PhysioNet EEGMMIDB-derived image, enter **160**.

For a different EEG source, enter the sampling frequency that was used to create the image. Do not guess it from the picture.


In [ ]:
FS = float(input("Enter sampling frequency in Hz (example: 160): "))

print("Sampling frequency:", FS, "Hz")


## 3. Extract the waveform from the image

The image is converted into a one-dimensional waveform by detecting the dark waveform pixels.


In [ ]:
def extract_waveform_from_image(image_path):
    image = Image.open(image_path).convert("L")
    image_array = np.array(image)

    # Dark pixels are assumed to belong to the waveform
    dark_pixels = image_array < 100

    height, width = dark_pixels.shape
    waveform = np.full(width, np.nan)

    for x in range(width):
        y_positions = np.where(dark_pixels[:, x])[0]

        if len(y_positions) > 0:
            waveform[x] = np.mean(y_positions)

    valid = ~np.isnan(waveform)

    if np.sum(valid) < 2:
        raise ValueError(
            "Could not detect the EEG waveform. "
            "Try an image with a clear dark waveform."
        )

    waveform = np.interp(
        np.arange(width),
        np.where(valid)[0],
        waveform[valid]
    )

    # Reverse image Y-axis
    waveform = -waveform

    # Remove DC component
    waveform = waveform - np.mean(waveform)

    # Normalize
    std = np.std(waveform)
    if std > 0:
        waveform = waveform / std

    return waveform


signal = extract_waveform_from_image(filename)

print("Waveform successfully extracted from image.")
print("Number of extracted samples:", len(signal))


In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(signal)
plt.title("Waveform Extracted From Input EEG Image")
plt.xlabel("Image Sample")
plt.ylabel("Normalized Amplitude")
plt.grid(True)

plt.show()


## 4. Define the band-pass filter

Frequency bands used:

- **Mu:** 8–13 Hz
- **Beta:** 13–30 Hz
- **Final preprocessing band:** 8–30 Hz


In [ ]:
def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    if low <= 0 or high >= 1:
        raise ValueError(
            f"Invalid frequency range {lowcut}–{highcut} Hz for "
            f"sampling frequency {fs} Hz."
        )

    b, a = butter(
        order,
        [low, high],
        btype="bandpass"
    )

    filtered_signal = filtfilt(
        b,
        a,
        signal
    )

    return filtered_signal


## 5. Apply Mu, Beta and Final band-pass filters


In [ ]:
# Mu band: 8–13 Hz
mu_signal = bandpass_filter(
    signal,
    lowcut=8,
    highcut=13,
    fs=FS
)

# Beta band: 13–30 Hz
beta_signal = bandpass_filter(
    signal,
    lowcut=13,
    highcut=30,
    fs=FS
)

# Final combined band: 8–30 Hz
final_signal = bandpass_filter(
    signal,
    lowcut=8,
    highcut=30,
    fs=FS
)

print("Filtering completed successfully.")


In [ ]:
def signal_to_image(signal, title, output_file):
    plt.figure(figsize=(12, 4))

    plt.plot(signal, linewidth=1)
    plt.title(title, fontsize=15)
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()


In [ ]:
mu_file = "EEG_Mu_8_13Hz.png"
beta_file = "EEG_Beta_13_30Hz.png"
final_file = "EEG_Final_8_30Hz.png"

signal_to_image(
    mu_signal,
    "Mu Band EEG (8–13 Hz)",
    mu_file
)

signal_to_image(
    beta_signal,
    "Beta Band EEG (13–30 Hz)",
    beta_file
)

signal_to_image(
    final_signal,
    "Final Preprocessed EEG (8–30 Hz)",
    final_file
)

print("Filtered EEG images saved.")


## 6. Display the complete preprocessing result


In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(15, 14)
)

# Original
axes[0].imshow(Image.open(filename))
axes[0].set_title(
    "1. ORIGINAL EEG IMAGE — INPUT",
    fontsize=16,
    fontweight="bold"
)
axes[0].axis("off")

# Mu
axes[1].imshow(Image.open(mu_file))
axes[1].set_title(
    "2. MU FILTERED EEG — 8–13 Hz",
    fontsize=16,
    fontweight="bold"
)
axes[1].axis("off")

# Beta
axes[2].imshow(Image.open(beta_file))
axes[2].set_title(
    "3. BETA FILTERED EEG — 13–30 Hz",
    fontsize=16,
    fontweight="bold"
)
axes[2].axis("off")

# Final
axes[3].imshow(Image.open(final_file))
axes[3].set_title(
    "4. FINAL PREPROCESSED EEG — 8–30 Hz",
    fontsize=16,
    fontweight="bold"
)
axes[3].axis("off")

plt.tight_layout()
plt.show()


## 7. Download the three processed images

Run each cell to download the corresponding result.


In [ ]:
files.download(mu_file)


In [ ]:
files.download(beta_file)


In [ ]:
files.download(final_file)


## Pipeline

**Manual EEG image upload → waveform extraction → Mu band-pass (8–13 Hz) → Beta band-pass (13–30 Hz) → Final band-pass (8–30 Hz) → preprocessed EEG images**

This notebook performs **preprocessing only**; no classifier is included.
